In [ ]:
import os
os.chdir('/project/galaxies') #TJ change working directory to be the parent directory
from Functions import *
_, filter_files = collect_M51_image_and_filter_files(filter_directory, image_directory)
from astropy.table import Table
from glob import glob
#TJ To create table of galaxies, run tjuchau/projects/EW_vs_Age/Create_cluster_table.py
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')
def get_EW_using_filters(feature_filter_file, continuum_filter_files, location, radius):
    #TJ load files

    #TJ get all 3 image fluxes
    Fnu_feature = get_image_flux(feature_filter_file, location, radius, replace_negatives=False)
    Fnu_cont = [get_image_flux(f, location, radius, replace_negatives=False) for f in continuum_filter_files]
    feature_filter = extract_filter_name(feature_filter_file)
    continuum_filters = [extract_filter_name(x) for x in continuum_filter_files]
    if Fnu_feature.unit != Fnu_cont[0].unit:
        print('units are not the same in the feature image and continuum image!')
    elif Fnu_feature.unit == u.W/(u.m**2*u.Hz):
            
        #TJ look up pivot wavelengths
        pivot_feat = jwst_pivots[feature_filter]
        pivot_cont = [jwst_pivots[extract_filter_name(f)] for f in continuum_filter_files]
        
        #TJ convert continuum levels into F_lambda using pivot wavelengths, still need to multiply by dlamda
        fλ_cont = [(Fnu * c / pivot**2).to(u.W / u.m**2 / u.m)
                   for Fnu, pivot in zip(Fnu_cont, pivot_cont)]
        
        #TJ get mean wavelengths
        cont_wls = [jwst_means[f] for f in continuum_filters]
        line_wl = jwst_means[feature_filter]
    
        #TJ interpolate continuum values to the feature wavelength
        feature_continuum = np.interp(
            line_wl.value,
            [w.value for w in cont_wls],
            [f.value for f in fλ_cont]
        ) * u.W / u.m**2 / u.m
        
        #TJ print continuum if needed
        #print("F_lamda of photo continuum : ", feature_continuum)
        #TJ get filter transmission curve info
        wl, T = get_filter_data(feature_filter)
    
        #TJ multiply feature F_lambda by dlambda to complete unit conversion
        norm = np.trapezoid(T, wl) / np.max(T)
        cont_in_filter = feature_continuum * norm
        
        #TJ convert feature filter's F_nu into F_lamda
        fλ_feature = ((Fnu_feature * c / pivot_feat**2).to(u.W / u.m**2 / u.m))*norm 
    
        #TJ feature area is only the area above the continuum
        feature_only = fλ_feature - cont_in_filter
    
        #TJEquivalent width is this area divided by the continuum level
        EW = (feature_only / feature_continuum).to(u.m)
    
        return EW
table[0]

In [ ]:
paEWs = []
for row in table:
    loc = [row['ra'], row['dec']]
    gal_name = row['galaxy']
    files = glob(f'tjuchau/data_files/JWST/images/{gal_name.upper()}/*')
    f150 = [x for x in files if extract_filter_name(x)=='F150W'][0]
    f187 = [x for x in files if extract_filter_name(x)=='F187N'][0]
    f300 = [x for x in files if extract_filter_name(x)=='F300M'][0]
    pa_EW = get_EW_using_filters(f187, [f150,f300], loc, 0.3*u.arcsec)
    paEWs.append(pa_EW)
table.add_column(paEWs, name = 'EW_187')


In [ ]:
plt.scatter(table['best.sfh.age'], table['EW_658'], s = 5, color = 'blue')
plt.scatter(table['best.sfh.age'], table['EW_187'], s = 5, color = 'red')
plt.xscale('log')
plt.yscale('log')

In [ ]:
import pickle
galaxy_tables['ngc1433'][7]['radius'] = 0.4
with open("/project/galaxies/tjuchau/data_files/misc_data/saved_data/Kiana_galaxies.pkl", "wb") as file:
    pickle.dump(galaxy_tables, file)

In [ ]:
young_clusters = kiana_table[kiana_table['EW_658'] > 8e-9]
image_files = glob.glob(f'tjuchau/data_files/JWST/images/NGC1433/*')
ages = []
ews = []
for row in young_clusters:
    ra = row['ra']
    dec = row['dec']
    age = row['best.sfh.age']
    ages.append(age)
    pa_EW = get_EW_using_filters(image_files[1], [image_files[0], image_files[2]], [ra,dec], 0.5*u.arcsec).value
    ews.append(pa_EW)
    print(f'Age : {age}, Pa-EW : {pa_EW}')
    try:
        show_images(image_files, [ra,dec], 0.5*u.arcsec, cmap = 'rainbow')
    except:
        print('Image does not cover this location')
plt.scatter(ages, ews)
plt.show()

In [ ]:
name = 'ngc1433'
ngc1433_bad = [0,4,6,9,10]
ngc1433_maybe = [7]
ngc1433_good = np.setdiff1d(np.arange(len(galaxy_tables[name])), np.concatenate([ngc1433_bad, ngc1433_maybe]))
image_files = glob.glob(f'tjuchau/data_files/JWST/images/{name.upper()}/*')
for i, row in enumerate(galaxy_tables[name][:100]):
    if i in ngc1433_good:
        location = row['location']
        print(i)
        show_images(image_files, location, row['radius']*u.arcsec, cmap = 'rainbow')

In [ ]:
for i, row in enumerate(galaxy_tables[name][ngc1433_bad]):
    location = row['location']
    print(ngc1433_bad[i])
    show_images(image_files, location, 0.3*u.arcsec, cmap = 'rainbow')